In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
!pip install trl

In [3]:
from trl import DPOConfig, DPOTrainer
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

2025-12-29 13:24:36.526676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767014676.709328      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767014676.761037      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767014677.185719      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767014677.185753      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767014677.185756      55 computation_placer.cc:177] computation placer alr

In [4]:
MODEL_ID = 'phuc-hoang1208/finetuned-vit5base-textsplitting'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map='auto',
)

ref_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map='auto',
)
ref_model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(36096, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(36096, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [5]:
train = load_dataset('vohuutridung/3190-stage2-data-v2', split='train')
validation = load_dataset('vohuutridung/3190-stage2-data-v2', split='validation')

print(train)
print(validation)

README.md:   0%|          | 0.00/183 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/11525 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/378 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 11525
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 378
})


# Train

In [7]:
args = DPOConfig(
    output_dir='./dpo',

    max_prompt_length=256,
    max_completion_length=256,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    warmup_ratio=0.1,

    logging_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,

    report_to='tensorboard',
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    processing_class=tokenizer,
    args=args,
    train_dataset=train,
    eval_dataset=validation,
    
)

Extracting prompt in train dataset:   0%|          | 0/11525 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/11525 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/11525 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/378 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/378 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/378 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [8]:
trainer.train()

Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
200,0.667100,0.619892,-0.045269,-0.222847,0.768229,0.177578,-20.456123,-28.174097,-35.680443,-35.160664
400,0.540000,0.526283,-0.219978,-0.795836,0.782292,0.575858,-22.203209,-33.903980,-36.310780,-35.758587
600,0.455800,0.484991,-0.359866,-1.235680,0.778125,0.875814,-23.602091,-38.302425,-37.030010,-36.455360
800,0.405100,0.455959,-0.459701,-1.588443,0.794271,1.128742,-24.600443,-41.830059,-37.606800,-37.018871
1000,0.372600,0.432150,-0.545685,-1.912477,0.796875,1.366792,-25.460281,-45.070389,-38.249668,-37.652508
1200,0.335700,0.417323,-0.659873,-2.257181,0.797917,1.597308,-26.602167,-48.517429,-38.929607,-38.332241
1400,0.333800,0.402625,-0.672042,-2.404749,0.800521,1.732707,-26.723848,-49.993118,-39.154850,-38.562027
1600,0.311900,0.395395,-0.763298,-2.652614,0.803125,1.889316,-27.636415,-52.471767,-39.681736,-39.090736
1800,0.303300,0.389426,-0.772870,-2.731124,0.795312,1.958254,-27.732134,-53.256863,-39.903816,-39.315342
2000,0.304800,0.385561,-0.774468,-2.769389,0.807292,1.994922,-27.748116,-53.639526,-39.975616,-39.387604


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2163, training_loss=0.39518281193282173, metrics={'train_runtime': 4106.4999, 'train_samples_per_second': 8.42, 'train_steps_per_second': 0.527, 'total_flos': 0.0, 'train_loss': 0.39518281193282173, 'epoch': 3.0})

In [9]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

REPO_ID = 'vohuutridung/vit5-base-split-sft-dpo-v2'
trainer.model.push_to_hub(REPO_ID)
trainer.processing_class.push_to_hub(REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/vohuutridung/vit5-base-split-sft-dpo-v2/commit/12145087266f85f5114a7b1d73cb66349b654451', commit_message='Upload tokenizer', commit_description='', oid='12145087266f85f5114a7b1d73cb66349b654451', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vohuutridung/vit5-base-split-sft-dpo-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='vohuutridung/vit5-base-split-sft-dpo-v2'), pr_revision=None, pr_num=None)